
# Illustrative sodium-cooled fast reactor (SFR) plant — ThunderGraph demo

**Context:** A compact **executable systems** sketch inspired by **pool-type sodium fast reactors** and integrated **decay heat removal**, **power conversion**, and **instrumentation → protection** paths — the kind of architecture advanced reactor developers care about. This is **not** a model of any specific commercial design or licensing basis; it is a **library demo** for `thundergraph-model`.

**Shows:** system-level **`requirement`** + **`allocate`** + acceptance **`expr`**, **`model.citation` + `model.references`** (Phase 8 fake provenance), **derived attributes** / **part constraints** (unitflow), **ports** + **`connect`** + **item flow**, discrete **behavior** (states, sequences), **`compile_graph` + `Evaluator`**, **`summarize_requirement_satisfaction`**, and **scenario trace** validation.

**Run:** From `thundergraph-model`: `uv sync --all-groups`, kernel = **`thundergraph-model/.venv`**. Architecture diagram: **`sfr_architecture.mmd`** / **`.md`**.




## 1. Plant-level requirements

| ID | Statement |
|----|-----------|
| **R-HL** | Primary sodium **hot-leg margin** shall remain within the analyzed operating envelope. |
| **R-DHR** | **Decay heat removal** capability shall cover post-shutdown decay load for the modeled snapshot. |
| **R-EXP** | **Net export** shall stay above the minimum dispatch floor when the unit is declared online. |
| **R-SDM** | **Cold shutdown reactivity margin** (normalized) shall exceed the minimum margin. |
| **R-PROT** | **Reactor protection** shall be ready to process excore telemetry (qualitative allocation). |

All **`model.requirement(...)`** declarations live on the **plant system** — not inside part `define` blocks. Use **`expr=`** where automated acceptance is desired (Phase 7).




## 2. Architecture (subsystem overview)

See **`sfr_architecture.md`** for a Mermaid figure. Subsystems in this slice:

- **Primary sodium loop** — hot-leg **ΔT** in **Kelvin** (ΔK and Δ°C are the same numerically for a difference) vs limit; part constraint + requirement mirror.
- **Decay heat removal** — removal capacity vs decay load (**W**).
- **Power island** — net export vs floor (two part constraints for the band; one requirement on the lower bound).
- **Reactivity / shutdown** — normalized shutdown margin (**dimensionless** stand-in for pcm / dollars in a real tool).
- **Core instrumentation → reactor protection** — structural **`connect`** with **`ReactorSignal`** item kind; **`emit_item`** drives the rack’s **`ReactorSignal`** event like the AEV perception → planner path.



In [1]:

# nbconvert may set cwd to `notebooks/` or monorepo root — find `tg_model` reliably.
from __future__ import annotations

from unitflow import Quantity
from unitflow.catalogs.si import W
from unitflow.core.dimensions import Dimension
from unitflow.core.scale import Scale
from unitflow.core.units import Unit

# SI catalog omits Kelvin today; define K for hot-leg ΔT (delta-K equals delta-°C numerically).
K = Unit(
    dimension=Dimension.base("temperature"),
    scale=Scale.one(),
    name="kelvin",
    symbol="K",
)

DIMLESS = Unit.dimensionless()

from tg_model.model.elements import Part, System
from tg_model.execution.configured_model import instantiate
from tg_model.execution.evaluator import Evaluator
from tg_model.execution.graph_compiler import compile_graph
from tg_model.execution.requirements import summarize_requirement_satisfaction
from tg_model.execution.run_context import RunContext
from tg_model.execution.validation import validate_graph
from tg_model.execution.instances import PartInstance
from tg_model.execution.behavior import (
    BehaviorTrace,
    behavior_authoring_projection,
    behavior_trace_to_records,
    dispatch_event,
    dispatch_sequence,
    emit_item,
    validate_scenario_trace,
)



In [2]:

def _reset_plant_types() -> None:
    for cls in (
        PrimarySodiumLoopPart,
        DecayHeatRemovalPart,
        PowerConversionPart,
        ReactivityControlPart,
        CoreInstrumentationPart,
        ReactorProtectionPart,
        SodiumCooledReactorPlant,
    ):
        cls._reset_compilation()


class PrimarySodiumLoopPart(Part):
    """Pool-type primary heat transport slice: hot-leg ΔT vs limit (Kelvin)."""

    @classmethod
    def define(cls, model):  # type: ignore[override]
        hot = model.parameter("hot_leg_delta", unit=K)
        hot_max = model.parameter("max_hot_leg_delta", unit=K)
        c_hot = model.constraint("hot_leg_within_limit", expr=hot <= hot_max)
        cite_ht = model.citation(
            "cite_primary_ht",
            title="Illustrative sodium hot-leg thermal basis (fake)",
            standard_id="DEMO-ASME-Nuclear-001",
        )
        model.references(c_hot, cite_ht)
        standby = model.state("standby", initial=True)
        circulating = model.state("circulating")
        start = model.event("start_pumps")
        model.transition(standby, circulating, on=start)


class DecayHeatRemovalPart(Part):
    """DRACS / auxiliary sodium loops abstracted: capacity vs decay heat (power)."""

    @classmethod
    def define(cls, model):  # type: ignore[override]
        cap = model.parameter("dhr_capacity_w", unit=W)
        decay = model.parameter("decay_heat_w", unit=W)
        c_dhr = model.constraint("dhr_covers_decay", expr=cap >= decay)
        cite_dhr = model.citation(
            "cite_decay_heat",
            title="Illustrative decay heat removal acceptance (fake)",
            standard_id="DEMO-Reg-Guide-1.206",
        )
        model.references(c_dhr, cite_dhr)


class PowerConversionPart(Part):
    """Steam / turbine island abstracted to net bus export (superheated cycle not modeled)."""

    @classmethod
    def define(cls, model):  # type: ignore[override]
        net = model.parameter("net_export_w", unit=W)
        lo = model.parameter("min_export_w", unit=W)
        hi = model.parameter("max_export_w", unit=W)
        model.constraint("export_ge_min", expr=net >= lo)
        model.constraint("export_le_max", expr=net <= hi)
        offline = model.state("offline", initial=True)
        online = model.state("online")
        sync = model.event("sync_grid")
        model.transition(offline, online, on=sync)


class ReactivityControlPart(Part):
    """Shutdown reactivity worth — normalized scalar for demo (real tools use pcm, $, etc.)."""

    @classmethod
    def define(cls, model):  # type: ignore[override]
        worth = model.parameter("cold_shutdown_margin", unit=DIMLESS)
        need = model.parameter("min_shutdown_margin", unit=DIMLESS)
        model.constraint("shutdown_margin_ok", expr=worth >= need)


class CoreInstrumentationPart(Part):
    """Excore sensors / flux mapping → telemetry port (item source)."""

    @classmethod
    def define(cls, model):  # type: ignore[override]
        model.port("excore_out", direction="out")
        model.item_kind("ReactorSignal")


class ReactorProtectionPart(Part):
    """Rack / ESFAS slice: consumes structural ReactorSignal; signal quality constraint."""

    @classmethod
    def define(cls, model):  # type: ignore[override]
        model.port("rack_in", direction="in")
        sig_q = model.parameter("last_signal_quality", unit=DIMLESS)
        min_q = model.parameter("min_signal_quality", unit=DIMLESS)
        model.constraint("signal_quality_ok", expr=sig_q >= min_q)

        def ingest(c: RunContext, p: PartInstance) -> None:
            pl = c.peek_item_payload(p.path_string, "ReactorSignal")
            if pl is not None:
                c.bind_input(p.last_signal_quality.stable_id, float(pl))

        model.action("ingest_signal", effect=ingest)
        ev = model.event("ReactorSignal")
        idle = model.state("monitoring", initial=True)
        latched = model.state("latched")
        model.transition(idle, latched, on=ev, effect="ingest_signal")
        model.action("verify_window", effect=lambda c, p: None)
        model.action("arm_scram", effect=lambda c, p: None)
        model.sequence("protection_window", steps=["verify_window", "arm_scram"])


class SodiumCooledReactorPlant(System):
    """Root plant: requirements, allocations, connect + startup scenario."""

    @classmethod
    def define(cls, model):  # type: ignore[override]
        primary = model.part("primary_loop", PrimarySodiumLoopPart)
        dhr = model.part("decay_heat", DecayHeatRemovalPart)
        conv = model.part("power_island", PowerConversionPart)
        rods = model.part("reactivity", ReactivityControlPart)
        ix = model.part("core_ix", CoreInstrumentationPart)
        rack = model.part("reactor_protection", ReactorProtectionPart)

        r_hl = model.requirement(
            "req_hot_leg",
            "Primary loop hot-leg margin shall stay within the analyzed envelope.",
            expr=primary.hot_leg_delta <= primary.max_hot_leg_delta,
        )
        r_dhr = model.requirement(
            "req_decay_heat",
            "Decay heat removal shall cover decay load for the modeled snapshot.",
            expr=dhr.dhr_capacity_w >= dhr.decay_heat_w,
        )
        r_exp = model.requirement(
            "req_export_floor",
            "Net export shall remain above minimum dispatch when online.",
            expr=conv.net_export_w >= conv.min_export_w,
        )
        r_sdm = model.requirement(
            "req_shutdown_margin",
            "Cold shutdown margin shall exceed the minimum.",
            expr=rods.cold_shutdown_margin >= rods.min_shutdown_margin,
        )
        r_prot = model.requirement(
            "req_protection_ready",
            "Reactor protection shall process excore telemetry (allocation only — no expr).",
        )

        model.allocate(r_hl, primary)
        model.allocate(r_dhr, dhr)
        model.allocate(r_exp, conv)
        model.allocate(r_sdm, rods)
        model.allocate(r_prot, rack)

        cite_gdc = model.citation(
            "cite_gdc",
            title="Illustrative GDC-style safety design criteria (fake)",
            standard_id="DEMO-NRC-GDC",
        )
        cite_ieee = model.citation(
            "cite_ieee_protection",
            title="Illustrative digital I&C / protection basis (fake)",
            standard_id="DEMO-IEEE-603",
        )
        model.references(r_hl, cite_gdc)
        model.references(r_dhr, cite_gdc)
        model.references(r_exp, cite_gdc)
        model.references(r_sdm, cite_gdc)
        model.references(r_prot, cite_ieee)
        model.references(rack.rack_in, cite_ieee)

        model.connect(source=ix.excore_out, target=rack.rack_in, carrying="ReactorSignal")

        model.scenario(
            "sfr_startup_snapshot",
            expected_event_order=[],
            expected_interaction_order=[
                ("primary_loop", "start_pumps"),
                ("power_island", "sync_grid"),
                ("reactor_protection", "ReactorSignal"),
            ],
            expected_item_kind_order=["ReactorSignal"],
        )


_reset_plant_types()




## 3. Constraints, derived quantities, and requirement acceptance

**Part** subsystems own **constraints**; the **plant** owns **requirements** with **`expr=`** where those acceptance checks are automated. One evaluation pass (**`compile_graph` → `evaluate`**) yields both part constraint results and requirement acceptance rows; **`summarize_requirement_satisfaction`** filters the latter.

**Phase 8:** Fake **`citation`** nodes and **`references`** (requirements, part constraints, and one **port**) show up in **`cm.references`** after **`instantiate`**.

`req_protection_ready` is **allocate-only** (no `expr`), so it does not add an automated acceptance check.



In [3]:

_reset_plant_types()
cm = instantiate(SodiumCooledReactorPlant)
graph, handlers = compile_graph(cm)
v = validate_graph(graph)
assert v.passed, v.failures

evalr = Evaluator(graph, compute_handlers=handlers)
ctxv = RunContext()
result = evalr.evaluate(
    ctxv,
    inputs={
        cm.primary_loop.hot_leg_delta.stable_id: Quantity(135, K),
        cm.primary_loop.max_hot_leg_delta.stable_id: Quantity(150, K),
        cm.decay_heat.dhr_capacity_w.stable_id: Quantity(32e6, W),
        cm.decay_heat.decay_heat_w.stable_id: Quantity(28e6, W),
        cm.power_island.net_export_w.stable_id: Quantity(320e6, W),
        cm.power_island.min_export_w.stable_id: Quantity(250e6, W),
        cm.power_island.max_export_w.stable_id: Quantity(345e6, W),
        cm.reactivity.cold_shutdown_margin.stable_id: 0.92,
        cm.reactivity.min_shutdown_margin.stable_id: 0.55,
        cm.reactor_protection.last_signal_quality.stable_id: 0.0,
        cm.reactor_protection.min_signal_quality.stable_id: 0.85,
    },
)

print("Constraint results (parts + requirement checks):", len(result.constraint_results))
for cr in result.constraint_results:
    print(" ", cr)

summary = summarize_requirement_satisfaction(result)
print(summary.check_count, "requirement acceptance checks | all_passed:", summary.all_passed)
for row in summary.results:
    tag = "PASS" if row.passed else "FAIL"
    print(" ", row.requirement_path, "→", row.allocation_target_path, "|", tag)

print("Provenance (model.references → citation):", len(cm.references))
for rb in cm.references:
    print(" ", rb)



Constraint results (parts + requirement checks): 10
  <ConstraintResult: SodiumCooledReactorPlant.decay_heat.dhr_covers_decay PASS>
  <ConstraintResult: reqcheck:SodiumCooledReactorPlant.req_decay_heat@1acf5a38-f004-51c8-abf3-1eb777b796dd PASS requirement='SodiumCooledReactorPlant.req_decay_heat' target='SodiumCooledReactorPlant.decay_heat'>
  <ConstraintResult: SodiumCooledReactorPlant.power_island.export_ge_min PASS>
  <ConstraintResult: SodiumCooledReactorPlant.power_island.export_le_max PASS>
  <ConstraintResult: reqcheck:SodiumCooledReactorPlant.req_export_floor@2c29eae6-d239-5e7a-9dae-2746de81b2cf PASS requirement='SodiumCooledReactorPlant.req_export_floor' target='SodiumCooledReactorPlant.power_island'>
  <ConstraintResult: SodiumCooledReactorPlant.primary_loop.hot_leg_within_limit PASS>
  <ConstraintResult: reqcheck:SodiumCooledReactorPlant.req_hot_leg@c0275077-5ca7-5bfc-ae48-e1d304a833ab PASS requirement='SodiumCooledReactorPlant.req_hot_leg' target='SodiumCooledReactorPlant.p


## 4. Compile structure



In [4]:

_reset_plant_types()
compiled = SodiumCooledReactorPlant.compile()
print("Owner:", compiled["owner"])
print("Parts:", [k for k, v in compiled["nodes"].items() if v["kind"] == "part"])
print("Edges:", len(compiled["edges"]))
for e in compiled["edges"]:
    print(" ", e["kind"], e.get("carrying") or "")



Owner: __main__.SodiumCooledReactorPlant
Parts: ['primary_loop', 'decay_heat', 'power_island', 'reactivity', 'core_ix', 'reactor_protection']
Edges: 12
  allocate 
  allocate 
  allocate 
  allocate 
  allocate 
  references 
  references 
  references 
  references 
  references 
  references 
  connect ReactorSignal



## 5. Protection authoring projection



In [5]:

_reset_plant_types()
proj = behavior_authoring_projection(ReactorProtectionPart)
print({k: proj[k] for k in ("sequences", "fork_joins", "actions")})



{'sequences': ['protection_window'], 'fork_joins': [], 'actions': ['arm_scram', 'ingest_signal', 'verify_window']}



## 6. Scripted startup + telemetry item (structural path)

Order: **start primary pumps** → **sync grid** → **`emit_item`** excore **`ReactorSignal`** to the rack (binds signal quality via effect). Then run the **protection** sequence.



In [6]:

_reset_plant_types()
cm = instantiate(SodiumCooledReactorPlant)
ctx = RunContext()
ctx.bind_input(cm.primary_loop.hot_leg_delta.stable_id, Quantity(120, K))
ctx.bind_input(cm.primary_loop.max_hot_leg_delta.stable_id, Quantity(150, K))
ctx.bind_input(cm.decay_heat.dhr_capacity_w.stable_id, Quantity(35e6, W))
ctx.bind_input(cm.decay_heat.decay_heat_w.stable_id, Quantity(25e6, W))
ctx.bind_input(cm.power_island.net_export_w.stable_id, Quantity(300e6, W))
ctx.bind_input(cm.power_island.min_export_w.stable_id, Quantity(200e6, W))
ctx.bind_input(cm.power_island.max_export_w.stable_id, Quantity(350e6, W))
ctx.bind_input(cm.reactivity.cold_shutdown_margin.stable_id, 0.88)
ctx.bind_input(cm.reactivity.min_shutdown_margin.stable_id, 0.5)
ctx.bind_input(cm.reactor_protection.last_signal_quality.stable_id, 0.0)
ctx.bind_input(cm.reactor_protection.min_signal_quality.stable_id, 0.8)

tr = BehaviorTrace()
r1 = dispatch_event(ctx, cm.primary_loop, "start_pumps", trace=tr)
r2 = dispatch_event(ctx, cm.power_island, "sync_grid", trace=tr)
out = emit_item(ctx, cm, cm.core_ix.excore_out, "ReactorSignal", 0.91, trace=tr)
dispatch_sequence(ctx, cm.reactor_protection, "protection_window", trace=tr)

print("Primary pumps:", r1.outcome)
print("Grid sync:", r2.outcome)
print("emit_item branches:", [x.outcome for x in out])
print("Rack signal quality:", ctx.get_value(cm.reactor_protection.last_signal_quality.stable_id))
print("Primary state:", ctx.get_active_behavior_state(cm.primary_loop.path_string))
print("Power island:", ctx.get_active_behavior_state(cm.power_island.path_string))



Primary pumps: fired
Grid sync: fired
emit_item branches: [<DispatchOutcome.FIRED: 'fired'>]
Rack signal quality: 0.91
Primary state: circulating
Power island: online



## 7. Scenario validation



In [7]:

_reset_plant_types()
cm = instantiate(SodiumCooledReactorPlant)
ctx = RunContext()
ctx.bind_input(cm.primary_loop.hot_leg_delta.stable_id, Quantity(120, K))
ctx.bind_input(cm.primary_loop.max_hot_leg_delta.stable_id, Quantity(150, K))
ctx.bind_input(cm.decay_heat.dhr_capacity_w.stable_id, Quantity(35e6, W))
ctx.bind_input(cm.decay_heat.decay_heat_w.stable_id, Quantity(25e6, W))
ctx.bind_input(cm.power_island.net_export_w.stable_id, Quantity(300e6, W))
ctx.bind_input(cm.power_island.min_export_w.stable_id, Quantity(200e6, W))
ctx.bind_input(cm.power_island.max_export_w.stable_id, Quantity(350e6, W))
ctx.bind_input(cm.reactivity.cold_shutdown_margin.stable_id, 0.88)
ctx.bind_input(cm.reactivity.min_shutdown_margin.stable_id, 0.5)
ctx.bind_input(cm.reactor_protection.last_signal_quality.stable_id, 0.0)
ctx.bind_input(cm.reactor_protection.min_signal_quality.stable_id, 0.8)

tr = BehaviorTrace()
dispatch_event(ctx, cm.primary_loop, "start_pumps", trace=tr)
dispatch_event(ctx, cm.power_island, "sync_grid", trace=tr)
emit_item(ctx, cm, cm.core_ix.excore_out, "ReactorSignal", 0.91, trace=tr)

ok, errors = validate_scenario_trace(
    definition_type=SodiumCooledReactorPlant,
    scenario_name="sfr_startup_snapshot",
    part_path=cm.path_string,
    trace=tr,
    root=cm.root,
)
print("Scenario OK:", ok)
print("Errors:", errors)



Scenario OK: True
Errors: []



## 8. Flat trace export



In [8]:

_reset_plant_types()
cm = instantiate(SodiumCooledReactorPlant)
ctx = RunContext()
ctx.bind_input(cm.primary_loop.hot_leg_delta.stable_id, Quantity(120, K))
ctx.bind_input(cm.primary_loop.max_hot_leg_delta.stable_id, Quantity(150, K))
ctx.bind_input(cm.decay_heat.dhr_capacity_w.stable_id, Quantity(35e6, W))
ctx.bind_input(cm.decay_heat.decay_heat_w.stable_id, Quantity(25e6, W))
ctx.bind_input(cm.power_island.net_export_w.stable_id, Quantity(300e6, W))
ctx.bind_input(cm.power_island.min_export_w.stable_id, Quantity(200e6, W))
ctx.bind_input(cm.power_island.max_export_w.stable_id, Quantity(350e6, W))
ctx.bind_input(cm.reactivity.cold_shutdown_margin.stable_id, 0.88)
ctx.bind_input(cm.reactivity.min_shutdown_margin.stable_id, 0.5)
ctx.bind_input(cm.reactor_protection.last_signal_quality.stable_id, 0.0)
ctx.bind_input(cm.reactor_protection.min_signal_quality.stable_id, 0.8)

tr = BehaviorTrace()
dispatch_event(ctx, cm.primary_loop, "start_pumps", trace=tr)
dispatch_event(ctx, cm.power_island, "sync_grid", trace=tr)
emit_item(ctx, cm, cm.core_ix.excore_out, "ReactorSignal", 0.91, trace=tr)

records = behavior_trace_to_records(tr)
for rec in records[:10]:
    print(rec)
print("...", len(records), "records")



{'kind': 'transition', 'step_index': 0, 'part_path': 'SodiumCooledReactorPlant.primary_loop', 'event_name': 'start_pumps', 'from_state': 'standby', 'to_state': 'circulating', 'effect_name': None}
{'kind': 'transition', 'step_index': 1, 'part_path': 'SodiumCooledReactorPlant.power_island', 'event_name': 'sync_grid', 'from_state': 'offline', 'to_state': 'online', 'effect_name': None}
{'kind': 'item_flow', 'step_index': 2, 'source_port_path': 'SodiumCooledReactorPlant.core_ix.excore_out', 'target_port_path': 'SodiumCooledReactorPlant.reactor_protection.rack_in', 'item_kind': 'ReactorSignal', 'payload': 0.91}
{'kind': 'transition', 'step_index': 3, 'part_path': 'SodiumCooledReactorPlant.reactor_protection', 'event_name': 'ReactorSignal', 'from_state': 'monitoring', 'to_state': 'latched', 'effect_name': 'ingest_signal'}
... 4 records
